In [0]:
# Objetivo:
# Importar as bibliotecas necessárias
# para leitura, manipulação e
# persistência dos dados.

# Justificativa:
# As bibliotecas importadas serão
# utilizadas ao longo de todo o
# processo de construção da camada
# Silver.

# Ação:
# Importa as bibliotecas utilizadas
# neste notebook.

import json
from pathlib import Path

import pandas as pd

In [0]:
# Objetivo:
# Definir as configurações utilizadas
# durante a execução do notebook.

# Justificativa:
# Os caminhos são centralizados no
# config.json e na silver_metadata,
# garantindo consistência entre as
# camadas Bronze e Silver.

# Ação:
# Carrega a configuração oficial do
# projeto e filtra os metadados da
# entidade processada neste notebook.

CONFIG_FILE_PATH = "/Volumes/workspace/default/vol_trio_drive/projetos/fiap/tech_challenge_fase2/config/config.json"

config = json.loads(
    Path(CONFIG_FILE_PATH).read_text(
        encoding="utf-8"
    )
)

BASE_PATH = Path(
    config["environment"]["base_path"]
)

CONFIG_PATH = Path(
    config["paths"]["config_path"]
)

LOG_PATH = Path(
    config["paths"]["log_path"]
)

EXECUTION_DATE = (
    config["project"]["execution_date"]
)

SILVER_METADATA_PATH = (
    CONFIG_PATH
    / "silver_metadata"
)

df_silver_metadata = pd.read_parquet(
    SILVER_METADATA_PATH
)

metadata_dataset = (
    df_silver_metadata[
        df_silver_metadata["dataset"]
        == "estados"
    ]
    .sort_values("ano")
    .reset_index(drop=True)
)

if set(metadata_dataset["ano"]) != {2023, 2024, 2025}:
    raise ValueError(
        "Metadados Silver incompletos "
        "para o dataset estados."
    )

CAMINHO_BRONZE = Path(
    metadata_dataset.iloc[0][
        "bronze_path"
    ]
).parent

CAMINHO_SILVER = Path(
    metadata_dataset.iloc[0][
        "silver_path"
    ]
).parent

print(
    "CAMINHO_BRONZE:",
    CAMINHO_BRONZE
)

print(
    "CAMINHO_SILVER:",
    CAMINHO_SILVER
)

display(metadata_dataset)

# 1. Auditoria da Fonte de Dados - Base Estados

> **Nota**
>
> Durante o desenvolvimento deste projeto foi utilizado o dicionário oficial
> dos Microdados da Avaliação da Alfabetização disponibilizado pelo INEP como
> referência para interpretação das variáveis, domínios e regras de negócio
> presentes nas bases de dados.

## 1.1 Leitura das Bases

**Contexto**

Os dados da entidade **Estados** foram disponibilizados pelo INEP em
arquivos distintos para os anos de **2023**, **2024** e **2025**, os quais
foram previamente organizados na camada Bronze e particionados por ano.

Nesta etapa, os partições são carregadas para o ambiente de análise,
preservando sua estrutura original para que seja possível realizar a
auditoria da qualidade dos dados antes da aplicação das transformações da
camada Silver.

**Objetivo**

Realizar a leitura das bases da entidade Estados referentes aos anos de
2023, 2024 e 2025.

**Resultado esperado**

Disponibilizar os dados dos três anos em DataFrames independentes,
mantendo a estrutura original dos arquivos da camada Bronze para as etapas
de auditoria e transformação da camada Silver.

### Integração com o Silver Orquestrador

Este notebook não utiliza caminhos locais ou nomes de arquivos fixos.

Os caminhos da entidade `estados` são obtidos da tabela:

```text
config/silver_metadata
```

A leitura é feita diretamente das partições Parquet da Bronze:

```text
bronze/estados/ano=2023
bronze/estados/ano=2024
bronze/estados/ano=2025
```

A lógica de auditoria, limpeza e transformação construída originalmente permanece preservada.

In [0]:
# Objetivo:
# Carregar as partições anuais da
# entidade estados na camada Bronze.

# Justificativa:
# A Bronze foi persistida em Parquet,
# organizada por entidade e ano.
# A Silver deve consumir diretamente
# essas partições governadas pelo
# silver_metadata.

# Ação:
# Lê as partições Bronze referentes
# aos anos de 2023, 2024 e 2025.

def caminho_bronze_ano(ano):
    registro = metadata_dataset[
        metadata_dataset["ano"] == ano
    ].iloc[0]

    return Path(
        registro["bronze_path"]
    )


df_estados_2023 = pd.read_parquet(
    caminho_bronze_ano(2023)
)

df_estados_2024 = pd.read_parquet(
    caminho_bronze_ano(2024)
)

df_estados_2025 = pd.read_parquet(
    caminho_bronze_ano(2025)
)

## 1.2 Inspeção Inicial da Estrutura

**Contexto**

Após o carregamento das bases da camada Bronze, realiza-se uma inspeção
inicial para compreender a estrutura dos dados disponibilizados pelo INEP
em cada ano da avaliação.

Essa verificação permite identificar a quantidade de colunas e possíveis
alterações estruturais entre as bases antes do processo de padronização da
camada Silver.

**Objetivo**

Inspecionar a estrutura das bases da entidade Estados referentes aos anos
de 2023, 2024 e 2025, identificando eventuais diferenças entre os esquemas
disponibilizados pelo INEP.

**Resultado esperado**

Obter uma visão inicial da estrutura das bases, permitindo identificar
alterações entre os anos e subsidiar as etapas de auditoria e
padronização da camada Silver.

In [0]:
# Objetivo:
# Realizar uma inspeção inicial da
# estrutura das bases de Estados.

# Justificativa:
# A inspeção inicial permite verificar
# o esquema das bases e identificar
# possíveis alterações nas colunas
# disponibilizadas pelo INEP ao longo
# dos anos da avaliação.

# Ação:
# Exibe a estrutura das bases de
# Estados para comparação entre os
# anos de 2023, 2024 e 2025.

print(df_estados_2023.columns.tolist())

print(df_estados_2024.columns.tolist())

print(df_estados_2025.columns.tolist())

## 1.3 Comparação dos Tipos de Dados

**Contexto**

Além da estrutura das bases, é importante verificar se os tipos de dados
das colunas permaneceram consistentes entre os anos da avaliação.

Alterações nos tipos podem impactar as etapas de transformação,
padronização e integração dos dados na camada Silver.

**Objetivo**

Comparar os tipos de dados das colunas presentes nas bases da entidade
Estados referentes aos anos de 2023, 2024 e 2025.

**Resultado esperado**

Identificar possíveis divergências entre os tipos de dados das colunas,
subsidiando a definição da estrutura padronizada da camada Silver.

In [0]:
# Objetivo:
# Comparar os tipos de dados das
# bases de estados dos anos de
# 2023, 2024 e 2025.

# Justificativa:
# Colunas com o mesmo nome podem
# apresentar tipos diferentes entre
# os anos ou novas variáveis podem
# ter sido incorporadas pelo INEP.

# Ação:
# Consolida os tipos de dados das
# três bases em um único relatório
# e classifica a compatibilidade
# entre os esquemas.

relatorio_dtypes = pd.DataFrame({
    "2023": df_estados_2023.dtypes.astype(str),
    "2024": df_estados_2024.dtypes.astype(str),
    "2025": df_estados_2025.dtypes.astype(str)
})

relatorio_dtypes = relatorio_dtypes.replace("nan", pd.NA)


def classificar_status(linha):

    if (
        pd.notna(linha["2023"])
        and pd.isna(linha["2024"])
        and pd.isna(linha["2025"])
    ):
        return "Exclusiva de 2023"

    if (
        pd.isna(linha["2023"])
        and pd.notna(linha["2024"])
        and pd.notna(linha["2025"])
    ):
        return "Nova a partir de 2024"

    if (
        pd.isna(linha["2023"])
        and pd.isna(linha["2024"])
        and pd.notna(linha["2025"])
    ):
        return "Nova em 2025"

    tipos = linha.dropna()

    if len(tipos.unique()) == 1:
        return "Compatível"

    return "Divergente"


relatorio_dtypes["status"] = relatorio_dtypes.apply(
    classificar_status,
    axis=1
)

display(relatorio_dtypes)

## 1.4 Análise dos Valores Ausentes

**Contexto**

Após a verificação da estrutura e dos tipos de dados, torna-se necessário
avaliar a presença de valores ausentes nas bases dos diferentes anos.

Essa análise permite identificar possíveis impactos sobre a qualidade dos
dados e definir se será necessário realizar algum tratamento antes da
persistência da camada Silver.

**Objetivo**

Identificar e quantificar a ocorrência de valores ausentes nas bases da
entidade Estados, avaliando a necessidade de tratamento durante o processo
de transformação.

**Resultado esperado**

Obter um diagnóstico da completude dos dados, permitindo justificar as
decisões adotadas em relação ao tratamento de valores ausentes na camada
Silver.

In [0]:
# Objetivo:
# Analisar a ocorrência de valores
# ausentes nas bases da entidade
# Estados.

# Justificativa:
# A identificação de valores ausentes
# permite avaliar a qualidade dos
# dados e definir eventuais ações de
# tratamento antes da persistência da
# camada Silver.

# Ação:
# Calcula a quantidade de valores
# ausentes por coluna para cada ano
# da avaliação.

for ano, df in [
    (2023, df_estados_2023),
    (2024, df_estados_2024),
    (2025, df_estados_2025)
]:

    print(f"\nValores ausentes - {ano}")

    valores_ausentes = (
        df.isna()
          .sum()
          .loc[lambda s: s > 0]
          .sort_values(ascending=False)
          .to_frame("Valores Ausentes")
    )

    if valores_ausentes.empty:
        print("Nenhum valor ausente encontrado.")
    else:
        display(valores_ausentes)

### 1.4.1 Análise dos Resultados

**Contexto**

Após a identificação dos valores ausentes, torna-se necessário interpretar
os resultados obtidos para verificar se as ocorrências representam
inconsistências na base de origem ou se decorrem das características dos
dados disponibilizados pelo INEP.

**Objetivo**

Interpretar os resultados da análise de valores ausentes e justificar as
decisões adotadas durante a construção da camada Silver.

**Resultado esperado**

Documentar que as bases originais de estados não apresentam valores
ausentes e registrar que as adaptações estruturais realizadas
posteriormente fazem parte do processo de padronização da camada Silver,
não caracterizando problemas de qualidade dos dados de origem.

## 1.5 Seleção das Colunas da Camada Silver

**Contexto**

A auditoria da estrutura das bases mostrou que o INEP ampliou a quantidade
de informações disponibilizadas a partir de 2024, incorporando colunas
relacionadas à distribuição dos estudantes por níveis de proficiência em
Língua Portuguesa.

Como essas informações possuem valor analítico para as etapas posteriores
do projeto, elas serão preservadas na camada Silver.

Para garantir a compatibilidade estrutural entre as partições anuais, as
colunas incorporadas pelo INEP a partir de 2024 serão adicionadas à base de
2023 durante o processo de padronização.

Nesta etapa são selecionadas todas as colunas que comporão a Base Estados
da camada Silver, assegurando que as bases dos anos de 2023, 2024 e 2025
compartilhem o mesmo esquema antes da persistência.

**Objetivo**

Definir o conjunto de atributos que comporá a Base Estados da camada
Silver, preservando as informações relevantes para as análises e etapas
posteriores do pipeline.

**Resultado esperado**

Obter três bases com a mesma estrutura de colunas, aptas para a
persistência particionada da camada Silver e para posterior integração na
camada Gold.

In [0]:
# Objetivo:
# Selecionar as colunas que farão
# parte da Base Estados da
# camada Silver.

# Justificativa:
# As bases de 2024 e 2025 passaram
# a disponibilizar colunas referentes
# aos níveis de proficiência em Língua
# Portuguesa. Essas informações possuem
# valor analítico e serão preservadas
# na camada Silver.

# Ação:
# Padroniza a estrutura das bases,
# criando as colunas ausentes na base
# de 2023 e selecionando o conjunto
# final de atributos da camada Silver.

colunas_silver = [
    "NU_ANO_AVALIACAO",
    "CO_UF",
    "SG_UF",
    "TP_SERIE",
    "ID_TIPO_REDE",
    "PC_ALUNO_ALFABETIZADO",
    "VL_MEDIA_LP",
    "PC_ALUNO_NIVEL_0_LP",
    "PC_ALUNO_NIVEL_1_LP",
    "PC_ALUNO_NIVEL_2_LP",
    "PC_ALUNO_NIVEL_3_LP",
    "PC_ALUNO_NIVEL_4_LP",
    "PC_ALUNO_NIVEL_5_LP",
    "PC_ALUNO_NIVEL_6_LP",
    "PC_ALUNO_NIVEL_7_LP",
    "PC_ALUNO_NIVEL_8_LP"
]

# Adiciona à base de 2023 as colunas
# introduzidas pelo INEP a partir de
# 2024, preservando a compatibilidade
# estrutural entre as partições.

for coluna in colunas_silver:
    if coluna not in df_estados_2023.columns:
        df_estados_2023[coluna] = pd.NA

df_estados_2023 = df_estados_2023[colunas_silver].copy()
df_estados_2024 = df_estados_2024[colunas_silver].copy()
df_estados_2025 = df_estados_2025[colunas_silver].copy()

## 1.6 Validação da Estrutura da Camada Silver

**Contexto**

Após a padronização das colunas, torna-se necessário verificar se as três
bases compartilham exatamente o mesmo esquema.

Essa validação garante que todas as partições anuais da camada Silver
possuam a mesma estrutura antes da persistência dos dados.

**Objetivo**

Validar a estrutura das bases da entidade Estados após a padronização das
colunas.

**Resultado esperado**

Confirmar que as bases de 2023, 2024 e 2025 possuem exatamente a mesma
quantidade de colunas, assegurando a compatibilidade estrutural entre as
partições da camada Silver.

In [0]:
# Objetivo:
# Validar a estrutura das bases da
# entidade Estados após o processo
# de padronização.

# Justificativa:
# A validação confirma que todas as
# partições da camada Silver possuem
# exatamente o mesmo esquema.

# Ação:
# Compara a quantidade de colunas das
# três bases e classifica o resultado
# da validação.

validacao = pd.DataFrame({
    "Base": ["2023", "2024", "2025"],
    "Quantidade de Colunas": [
        len(df_estados_2023.columns),
        len(df_estados_2024.columns),
        len(df_estados_2025.columns)
    ]
})

quantidade_esperada = len(colunas_silver)

validacao["Status"] = validacao[
    "Quantidade de Colunas"
].apply(
    lambda x: (
        "Compatível"
        if x == quantidade_esperada
        else "Incompatível"
    )
)

display(validacao)

## 1.7 Persistência da Base Estados

**Contexto**

Após a validação da estrutura das bases, os dados encontram-se prontos para
serem persistidos na camada Silver.

Nesta etapa, cada partição anual é armazenada em seu respectivo diretório,
preservando a organização por entidade e por ano definida para a
arquitetura do projeto.

**Objetivo**

Persistir as bases da entidade Estados na camada Silver, mantendo o
particionamento anual.

**Resultado esperado**

Armazenar as bases tratadas da entidade Estados na camada Silver,
preservando a estrutura padronizada e a organização do Data Lake para as
etapas posteriores do pipeline analítico.

In [0]:
# Objetivo:
# Persistir as bases da entidade
# estados na camada Silver.

# Justificativa:
# A persistência utiliza os caminhos
# e nomes de arquivo registrados na
# silver_metadata, eliminando caminhos
# fixos e mantendo o particionamento
# anual definido no setup.

# Ação:
# Grava as bases tratadas dos anos de
# 2023, 2024 e 2025 em CSV UTF-8.

for ano, df in [
    (2023, df_estados_2023),
    (2024, df_estados_2024),
    (2025, df_estados_2025)
]:
    registro = metadata_dataset[
        metadata_dataset["ano"] == ano
    ].iloc[0]

    destino = Path(
        registro["silver_path"]
    )

    nome_arquivo = registro[
        "silver_file_name"
    ]

    destino.mkdir(
        parents=True,
        exist_ok=True
    )

    df.to_csv(
        destino / nome_arquivo,
        sep=";",
        decimal=",",
        encoding="utf-8",
        index=False
    )

    print(
        f"Silver salva: "
        f"{destino / nome_arquivo}"
    )

# Conclusão

Ao longo deste notebook foi realizada a auditoria estrutural das bases da
entidade **Estados** referentes aos anos de **2023**, **2024** e
**2025**, identificando a evolução do esquema de dados disponibilizado pelo
INEP.

A auditoria contemplou a verificação da estrutura das bases, a comparação
dos tipos de dados e a análise dos valores ausentes. Não foram
identificadas inconsistências relacionadas à completude dos dados nas bases
originais disponibilizadas pelo INEP, não sendo necessário aplicar
tratamentos de imputação ou exclusão de registros durante essa etapa.

A análise evidenciou que, a partir de 2024, foram incorporadas novas
colunas relacionadas à distribuição percentual dos estudantes por níveis de
proficiência em Língua Portuguesa. Considerando o potencial analítico
dessas informações, optou-se por preservá-las na camada Silver, realizando
a padronização do esquema por meio da inclusão dessas colunas na base de
2023.

Após a padronização das colunas e validação da estrutura, as bases foram
persistidas na camada **Silver**, mantendo a organização por entidade e o
particionamento por ano, conforme a arquitetura definida para o projeto.

Dessa forma, a Base Estados da camada Silver passa a representar uma
versão padronizada, auditada e organizada dos dados consolidados por
unidade da federação, constituindo uma fonte confiável para a construção
dos indicadores analíticos, das comparações entre metas e resultados e das
análises temporais desenvolvidas na camada Gold.